## §1 Setup & Imports

This notebook trains and compares six ensemble methods from class
(Bagging/RF, AdaBoost, GradientBoosting, XGBoost, Voting, Stacking)
plus a Decision Tree baseline. Hyperparameters are optimised with Optuna
(MedianPruner, 50 trials, macro F1 on val set). The test set is used
only once per model for final evaluation in §12.
Hardware: Ryzen 9800X3D (n_jobs=-1) for sklearn models,
RTX 5070 Ti (device='cuda') for XGBoost.

In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, classification_report,
    ConfusionMatrixDisplay)
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED          = 42
N_TRIALS      = 50
PROCESSED_DIR = '../data/processed/'
CLASS_NAMES   = ['Benign','BruteForce','DDoS','DoS','Mirai','Recon','Spoofing','Web-based']

def load_split(name):
    df = pd.read_parquet(PROCESSED_DIR + name)
    return df.drop(columns='y').values, df['y'].values

X_train, y_train = load_split('train.parquet')
X_val,   y_val   = load_split('val.parquet')
X_test,  y_test  = load_split('test.parquet')

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

results_val  = []
results_test = []

# Checkpoint: si existe, carga params de RF y AdaBoost ya optimizados
_ckpt = PROCESSED_DIR + 'checkpoint_after_ada.json'
if os.path.exists(_ckpt):
    _data           = json.load(open(_ckpt))
    rf_best_params  = _data['rf_best_params']
    ada_best_params = _data['ada_best_params']
    ada_base_depth  = _data['ada_base_depth']
    results_val     = _data['results_val_partial']
    print()
    print('CHECKPOINT CARGADO — salta las celdas §4, §5, §6 y ejecuta desde §7')
    print(f'  rf_best_params : {rf_best_params}')
    print(f'  ada_best_params: {ada_best_params}')
    print(f'  ada_base_depth : {ada_base_depth}')
    print(f'  results_val    : {[r["Model"] for r in results_val]}')

## §2 Resampling & Speed Subsamples

RandomOverSampler was selected in notebook 02 as the best imbalance strategy
(macro F1: 0.8247 with a DecisionTree probe). It duplicates minority samples
until all classes reach the majority count — no synthetic interpolation.

Two speed subsamples are derived from X_train_res:
- **X_opt (100K)**: used inside every Optuna objective. Hyperparameter search
  does not require the full dataset; a representative stratified subsample
  is sufficient. The final model always retrains on the larger set.
- **X_final_gbm (500K)**: used only for GradientBoostingClassifier's final model.
  Unlike RF and XGBoost, sklearn GBM is sequential (no tree-level parallelism),
  so training on 4.2M rows is impractical. 500K preserves class balance and
  provides sufficient signal for convergence. All other final models use the
  full 4.2M-row X_train_res.

In [8]:
ros = RandomOverSampler(random_state=SEED)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

_, X_opt, _, y_opt = train_test_split(
    X_train_res, y_train_res,
    test_size=100_000, stratify=y_train_res, random_state=SEED
)

_, X_final_gbm, _, y_final_gbm = train_test_split(
    X_train_res, y_train_res,
    test_size=500_000, stratify=y_train_res, random_state=SEED
)

print(f'Full resampled  : {X_train_res.shape[0]:>10,} rows')
print(f'Optuna subsample: {X_opt.shape[0]:>10,} rows  (HPT trials, all models)')
print(f'GBM subsample   : {X_final_gbm.shape[0]:>10,} rows  (GBM final model only)')
print()

print(f'  {"Class":<15} {"Before":>10} {"After":>10}')
print('  ' + '-'*38)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<15} {int((y_train==i).sum()):>10,} {int((y_train_res==i).sum()):>10,}')

Full resampled  :  4,234,192 rows
Optuna subsample:    100,000 rows  (HPT trials, all models)
GBM subsample   :    500,000 rows  (GBM final model only)

  Class               Before      After
  --------------------------------------
  Benign              17,035    529,274
  BruteForce             118    529,274
  DDoS               529,274    529,274
  DoS                125,990    529,274
  Mirai               13,841    529,274
  Recon                5,442    529,274
  Spoofing             7,571    529,274
  Web-based              355    529,274


## §3 Helper: evaluate_model()

Centralising metric collection ensures consistent reporting across all models.
MCC (Matthews Correlation Coefficient) is included alongside macro F1 as a
secondary metric robust to class imbalance — it accounts for all four cells
of the confusion matrix and is informative when class sizes differ by orders
of magnitude.

In [9]:
def evaluate_model(name, model, X_eval, y_eval, results):
    y_pred = model.predict(X_eval)
    row = {
        'Model':    name,
        'Accuracy': round(accuracy_score(y_eval, y_pred), 4),
        'Macro F1': round(f1_score(y_eval, y_pred, average='macro', zero_division=0), 4),
        'Macro P':  round(precision_score(y_eval, y_pred, average='macro', zero_division=0), 4),
        'Macro R':  round(recall_score(y_eval, y_pred, average='macro', zero_division=0), 4),
        'MCC':      round(matthews_corrcoef(y_eval, y_pred), 4),
    }
    results.append(row)
    for k, v in row.items():
        if k != 'Model':
            print(f'  {k:<12}: {v:.4f}')
    return row

## §4 Baseline — Decision Tree

The Decision Tree baseline reproduces the best result from notebook 02
(RandomOverSampler + DT probe, macro F1 ~0.82) as a reference.
Every ensemble method should exceed this — that is precisely the purpose
of ensembling: combining weak learners to produce a stronger one.
No Optuna tuning is applied to the baseline.

In [10]:
dt = DecisionTreeClassifier(random_state=SEED)
dt.fit(X_train_res, y_train_res)
print('Decision Tree (val):')
evaluate_model('Decision Tree', dt, X_val, y_val, results_val)
print()
print(classification_report(y_val, dt.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

Decision Tree (val):
  Accuracy    : 0.9936
  Macro F1    : 0.8247
  Macro P     : 0.8233
  Macro R     : 0.8263
  MCC         : 0.9839

              precision    recall  f1-score   support

      Benign       0.91      0.91      0.91      3650
  BruteForce       0.54      0.54      0.54        26
        DDoS       1.00      1.00      1.00    113416
         DoS       1.00      1.00      1.00     26997
       Mirai       1.00      1.00      1.00      2966
       Recon       0.80      0.79      0.79      1166
    Spoofing       0.83      0.82      0.83      1623
   Web-based       0.51      0.55      0.53        76

    accuracy                           0.99    149920
   macro avg       0.82      0.83      0.82    149920
weighted avg       0.99      0.99      0.99    149920



## §5 Random Forest (Bagging)

Random Forest is the canonical Bagging ensemble (class notebook 2_1).
Each tree is trained on a bootstrap sample; predictions are aggregated
by majority vote. n_jobs=-1 parallelises tree construction across all
16 cores of the Ryzen 9800X3D — wall-clock time scales roughly as 1/n_cores.
Optuna trials use X_opt (100K); the final model trains on the full 4.2M
where n_jobs=-1 again provides the speedup.

In [11]:
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth':         trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    model = RandomForestClassifier(**params, n_jobs=-1, random_state=SEED)
    model.fit(X_opt, y_opt)
    return f1_score(y_val, model.predict(X_val), average='macro')

study_rf = optuna.create_study(direction='maximize',
                                pruner=optuna.pruners.MedianPruner())
study_rf.optimize(objective_rf, n_trials=N_TRIALS)
rf_best_params = study_rf.best_params
print('Best params:', rf_best_params)
print('Best macro F1 (val):', study_rf.best_value)
print()

rf = RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)
rf.fit(X_train_res, y_train_res)
print('Random Forest (val):')
evaluate_model('Random Forest', rf, X_val, y_val, results_val)
print()
print(classification_report(y_val, rf.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

Best params: {'n_estimators': 50, 'max_depth': 25, 'min_samples_split': 2, 'max_features': 'sqrt'}
Best macro F1 (val): 0.8153420728869343

Random Forest (val):
  Accuracy    : 0.9948
  Macro F1    : 0.8134
  Macro P     : 0.9232
  Macro R     : 0.7735
  MCC         : 0.9868

              precision    recall  f1-score   support

      Benign       0.90      0.97      0.93      3650
  BruteForce       1.00      0.31      0.47        26
        DDoS       1.00      1.00      1.00    113416
         DoS       1.00      1.00      1.00     26997
       Mirai       1.00      0.99      1.00      2966
       Recon       0.86      0.78      0.82      1166
    Spoofing       0.89      0.83      0.86      1623
   Web-based       0.74      0.30      0.43        76

    accuracy                           0.99    149920
   macro avg       0.92      0.77      0.81    149920
weighted avg       0.99      0.99      0.99    149920



## §6 AdaBoost (Boosting)

AdaBoost trains estimators sequentially, upweighting misclassified samples
at each step (class notebook 2_1). The base estimator is a shallow Decision
Tree (max_depth 1–5); deeper bases risk overfitting when boosted.
learning_rate controls shrinkage — smaller values require more estimators
but generally generalise better. AdaBoost does not support n_jobs for
sequential boosting, but each base estimator is shallow so individual
fits are fast.

In [12]:
def objective_ada(trial):
    max_depth = trial.suggest_int('max_depth', 1, 5)
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 300, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
    }
    base  = DecisionTreeClassifier(max_depth=max_depth)
    model = AdaBoostClassifier(estimator=base, **params, random_state=SEED)
    model.fit(X_opt, y_opt)
    return f1_score(y_val, model.predict(X_val), average='macro')

study_ada = optuna.create_study(direction='maximize',
                                 pruner=optuna.pruners.MedianPruner())
study_ada.optimize(objective_ada, n_trials=N_TRIALS)
ada_best_params = study_ada.best_params.copy()
ada_base_depth  = ada_best_params.pop('max_depth')
print('Best params:', ada_best_params, '| base max_depth:', ada_base_depth)
print('Best macro F1 (val):', study_ada.best_value)
print()

ada_base = DecisionTreeClassifier(max_depth=ada_base_depth)
ada = AdaBoostClassifier(estimator=ada_base, **ada_best_params, random_state=SEED)
ada.fit(X_train_res, y_train_res)
print('AdaBoost (val):')
evaluate_model('AdaBoost', ada, X_val, y_val, results_val)
print()
print(classification_report(y_val, ada.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

Best params: {'n_estimators': 250, 'learning_rate': 0.2725843888825499} | base max_depth: 5
Best macro F1 (val): 0.8550235091671528

AdaBoost (val):
  Accuracy    : 0.9942
  Macro F1    : 0.8546
  Macro P     : 0.8903
  Macro R     : 0.8374
  MCC         : 0.9852

              precision    recall  f1-score   support

      Benign       0.94      0.89      0.91      3650
  BruteForce       1.00      0.54      0.70        26
        DDoS       1.00      1.00      1.00    113416
         DoS       1.00      1.00      1.00     26997
       Mirai       1.00      1.00      1.00      2966
       Recon       0.77      0.86      0.81      1166
    Spoofing       0.82      0.86      0.84      1623
   Web-based       0.60      0.55      0.58        76

    accuracy                           0.99    149920
   macro avg       0.89      0.84      0.85    149920
weighted avg       0.99      0.99      0.99    149920



In [14]:
  import json

  checkpoint = {
      'rf_best_params':    rf_best_params,
      'ada_best_params':   ada_best_params,
      'ada_base_depth':    int(ada_base_depth),
      'results_val_partial': results_val,
  }

  with open('../data/processed/checkpoint_after_ada.json', 'w') as f:
      json.dump(checkpoint, f, indent=2)

  print('Guardado:', list(checkpoint.keys()))
  print('Modelos en results_val:', [r['Model'] for r in results_val])

Guardado: ['rf_best_params', 'ada_best_params', 'ada_base_depth', 'results_val_partial']
Modelos en results_val: ['Decision Tree', 'Random Forest', 'AdaBoost']


## §7 Gradient Boosting (Boosting)

GradientBoosting minimises a differentiable loss on pseudo-residuals
sequentially (class notebook 2_1). subsample < 1.0 implements Stochastic
Gradient Boosting, which acts as regularisation. Unlike RF, sklearn's
GradientBoostingClassifier does not parallelise across trees — training
time scales linearly with n_estimators × dataset_size. For this reason,
the final model uses the 500K subsample (X_final_gbm) rather than the
full 4.2M. The Optuna search on X_opt already identified robust
hyperparameters; the final fit on 500K provides sufficient training signal.

In [13]:
def objective_gb(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 200, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 2, 6),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = GradientBoostingClassifier(**params, random_state=SEED)
    model.fit(X_opt, y_opt)
    return f1_score(y_val, model.predict(X_val), average='macro')

study_gb = optuna.create_study(direction='maximize',
                                pruner=optuna.pruners.MedianPruner())
study_gb.optimize(objective_gb, n_trials=N_TRIALS)
gb_best_params = study_gb.best_params
print('Best params:', gb_best_params)
print('Best macro F1 (val):', study_gb.best_value)
print()

gb = GradientBoostingClassifier(**gb_best_params, random_state=SEED)
gb.fit(X_final_gbm, y_final_gbm)
print('Gradient Boosting (val):')
evaluate_model('Gradient Boosting', gb, X_val, y_val, results_val)
print()
print(classification_report(y_val, gb.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

[W 2026-03-23 17:01:01,188] Trial 12 failed with parameters: {'n_estimators': 150, 'learning_rate': 0.29459096362266357, 'max_depth': 6, 'subsample': 0.8304497160616828} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/eeguskiza/iot-intrusion-detection/venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_5693/1943988288.py", line 9, in objective_gb
    model.fit(X_opt, y_opt)
  File "/home/eeguskiza/iot-intrusion-detection/venv/lib/python3.10/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/eeguskiza/iot-intrusion-detection/venv/lib/python3.10/site-packages/sklearn/ensemble/_gb.py", line 787, in fit
    n_stages = self._fit_stages(
  File "/home/eeguskiza/iot-intrusion-detection/venv/lib/python3.10/site-packages/sklearn/ensemble/_gb.py", line 883, in _fit_stages
    raw_predict

KeyboardInterrupt: 

## §8 XGBoost (GPU Boosting)

XGBoost extends GradientBoosting with L1/L2 regularisation (reg_alpha,
reg_lambda), column subsampling (colsample_bytree), and histogram-based
split finding (tree_method='hist'). device='cuda' offloads tree construction
to the RTX 5070 Ti (16GB VRAM), making XGBoost the fastest model in this
notebook — both in Optuna trials and in final training on the full 4.2M rows.
This is the key reason XGBoost can use the full resampled set while sklearn
GBM cannot. Per Anis (2024), XGBoost achieves >98% macro F1 on CICIoT2023
with additional feature selection.

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500, step=100),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    model = xgb.XGBClassifier(
        **params,
        tree_method='hist', device='cuda',
        eval_metric='mlogloss', verbosity=0, random_state=SEED,
    )
    model.fit(X_opt, y_opt)
    return f1_score(y_val, model.predict(X_val), average='macro')

study_xgb = optuna.create_study(direction='maximize',
                                 pruner=optuna.pruners.MedianPruner())
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
xgb_best_params = study_xgb.best_params
print('Best params:', xgb_best_params)
print('Best macro F1 (val):', study_xgb.best_value)
print()

xgb_model = xgb.XGBClassifier(
    **xgb_best_params,
    tree_method='hist', device='cuda',
    eval_metric='mlogloss', verbosity=0, random_state=SEED,
)
xgb_model.fit(X_train_res, y_train_res)
print('XGBoost (val):')
evaluate_model('XGBoost', xgb_model, X_val, y_val, results_val)
print()
print(classification_report(y_val, xgb_model.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

## §9 Voting Classifier

VotingClassifier aggregates predictions from heterogeneous base models
(class notebook 2_1). Hard voting takes the majority class label; soft
voting averages predicted probabilities — typically superior when models
are well-calibrated. The ensemble reuses already-tuned RF and XGBoost
parameters plus an untuned DT, providing complementary diversity:
bagging (RF) + boosting (XGBoost) + single tree (DT). n_jobs=-1 on the
VotingClassifier parallelises the fit of each base estimator independently.

In [ ]:
def make_estimators():
    return [
        ('rf',  RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**xgb_best_params, tree_method='hist',
                                   device='cuda', verbosity=0, random_state=SEED)),
        ('dt',  DecisionTreeClassifier(random_state=SEED)),
    ]

vote_hard = VotingClassifier(estimators=make_estimators(), voting='hard', n_jobs=-1)
vote_hard.fit(X_train_res, y_train_res)
print('Voting Hard (val):')
h = evaluate_model('Voting (hard)', vote_hard, X_val, y_val, results_val)

vote_soft = VotingClassifier(estimators=make_estimators(), voting='soft', n_jobs=-1)
vote_soft.fit(X_train_res, y_train_res)
print()
print('Voting Soft (val):')
s = evaluate_model('Voting (soft)', vote_soft, X_val, y_val, results_val)

print()
print(f'  {"Metric":<12}  {"Hard":>8}  {"Soft":>8}')
print('  ' + '-'*32)
for k in ['Accuracy','Macro F1','Macro P','Macro R','MCC']:
    print(f'  {k:<12}  {h[k]:>8.4f}  {s[k]:>8.4f}')

## §10 Stacking Classifier

StackingClassifier (class notebook 2_1) trains base estimators on the
full training set, generates out-of-fold predictions via cv=5, then
trains a meta-learner (LogisticRegression) on those predictions.
This allows the meta-learner to learn which base model to trust for each
region of the feature space. Base models: RF (bagging), XGBoost (boosting),
AdaBoost (adaptive boosting) — three distinct inductive biases.
LogisticRegression as meta-learner is the standard pattern from class
and prevents overfitting at the stacking level.
n_jobs=-1 parallelises base model fitting across CV folds.

In [ ]:
ada_base_for_stack = DecisionTreeClassifier(max_depth=ada_base_depth)
ada_for_stack = AdaBoostClassifier(
    estimator=ada_base_for_stack, **ada_best_params, random_state=SEED
)

base_estimators = [
    ('rf',  RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)),
    ('xgb', xgb.XGBClassifier(**xgb_best_params, tree_method='hist',
                                device='cuda', verbosity=0, random_state=SEED)),
    ('ada', ada_for_stack),
]

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1,
)
stack.fit(X_train_res, y_train_res)
print('Stacking (val):')
evaluate_model('Stacking', stack, X_val, y_val, results_val)
print()
print(classification_report(y_val, stack.predict(X_val),
      target_names=CLASS_NAMES, zero_division=0))

## §11 Comparative Results Table (Validation Set)

The validation table ranks models by macro F1 — the primary metric.
Validation results guide model selection and are used by Optuna; test
results (§12) are the final reported performance. A model with high
validation F1 but lower test F1 indicates overfit to the validation set
during Optuna's 50-trial search.

In [ ]:
df_val = pd.DataFrame(results_val).set_index('Model')
best_model_name = df_val['Macro F1'].idxmax()

print('=== Validation Results ===')
print(df_val.to_string())
print(f'\nBest model (val Macro F1): {best_model_name} — {df_val.loc[best_model_name,"Macro F1"]:.4f}')
print()

metrics = ['Accuracy','Macro F1','Macro P','Macro R','MCC']
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(df_val))
width = 0.15
for j, m in enumerate(metrics):
    ax.bar(x + j*width, df_val[m], width, label=m)
ax.set_xticks(x + width*2)
ax.set_xticklabels(df_val.index, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Validation Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## §12 Final Evaluation on Test Set

The test set is used here for the first and only time — it was not seen
during training or Optuna tuning. Comparing val and test results detects
overfitting to the validation set. The confusion matrix for the best model
exposes which class pairs are mutually confused; expected confusions are
Web-based ↔ Benign and BruteForce ↔ Recon due to overlapping traffic patterns
documented in the original paper (Neto et al., 2023).

In [ ]:
models_to_eval = [
    ('Decision Tree',     dt),
    ('Random Forest',     rf),
    ('AdaBoost',          ada),
    ('Gradient Boosting', gb),
    ('XGBoost',           xgb_model),
    ('Voting (hard)',     vote_hard),
    ('Voting (soft)',     vote_soft),
    ('Stacking',          stack),
]

for name, model in models_to_eval:
    print(f'{name}:')
    evaluate_model(name, model, X_test, y_test, results_test)
    print()

df_test = pd.DataFrame(results_test).set_index('Model')
best_test_name = df_test['Macro F1'].idxmax()

print('=== Test Results ===')
print(df_test.sort_values('Macro F1', ascending=False).to_string())
print(f'\nBest model (test Macro F1): {best_test_name}')
print()

best_model_obj = dict(models_to_eval)[best_test_name]
y_pred_best = best_model_obj.predict(X_test)

print(f'=== {best_test_name} — Classification Report ===')
print(classification_report(y_test, y_pred_best,
      target_names=CLASS_NAMES, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=CLASS_NAMES,
    normalize='true', cmap='Blues',
    xticks_rotation=45, ax=ax
)
ax.set_title(f'Confusion Matrix — {best_test_name} (normalised, test set)')
plt.tight_layout()
plt.show()

## §13 Export Results for Notebook 04

Metrics and hyperparameters are saved to CSV/JSON so that notebook 04
loads them independently. No model objects are serialised — notebook 04
performs only analysis and visualisation, requiring no sklearn/xgboost
at runtime.

In [ ]:
df_val.reset_index().to_csv(PROCESSED_DIR + 'results_val.csv', index=False)
df_test.reset_index().to_csv(PROCESSED_DIR + 'results_test.csv', index=False)

best_hparams = {
    'RandomForest':     rf_best_params,
    'AdaBoost':         {**ada_best_params, 'base_max_depth': ada_base_depth},
    'GradientBoosting': gb_best_params,
    'XGBoost':          xgb_best_params,
}
with open(PROCESSED_DIR + 'best_hparams.json', 'w') as f:
    json.dump(best_hparams, f, indent=2)

with open(PROCESSED_DIR + 'best_model_name.json', 'w') as f:
    json.dump({'val_best': best_model_name, 'test_best': best_test_name}, f, indent=2)

print('Exported:')
for fname in ['results_val.csv','results_test.csv','best_hparams.json','best_model_name.json']:
    size = os.path.getsize(PROCESSED_DIR + fname)
    print(f'  {fname:<35} {size:,} bytes')

## §14 Summary

### Model Ranking (populated at runtime)

| Model | Val Macro F1 | Test Macro F1 |
|---|---|---|
| Best model | see §11 | see §12 |
| ... | ... | ... |

### Key observations
- The winning model is determined by test macro F1 in §12.
- If val ≈ test, Optuna did not overfit to the validation set across 50 trials.
- BruteForce and Web-based remain the hardest classes due to low sample counts
  (~118 and ~507 training samples respectively) even after RandomOverSampler.
- XGBoost (device='cuda') trains on the full 4.2M rows in minutes;
  GradientBoostingClassifier requires a 500K subsample for the same budget.

### Handoff to notebook 04
Notebook 04 loads results_val.csv, results_test.csv, best_hparams.json, and
best_model_name.json for final analysis, visualisation, and discussion.
No model objects are required.